In [1]:
import numpy as np

X = np.load(
    "../embeddings/amp_embeddings.npy"
)

print(X.shape)
print(X.dtype)

(8400, 1280)
float32


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_val = train_test_split(
    X,
    test_size=0.1,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

(7560, 1280)
(840, 1280)


In [3]:
print(X.shape)
print(X.dtype)
print(X_train.shape)
print(X_val.shape)

(8400, 1280)
float32
(7560, 1280)
(840, 1280)


In [4]:
import torch
import torch.nn as nn

class AMPAutoencoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(1280, 1024),
            nn.ReLU(),

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 512),
            nn.ReLU(),

            nn.Linear(512, 1024),
            nn.ReLU(),

            nn.Linear(1024, 1280)
        )

    def forward(self, x):

        z = self.encoder(x)

        x_hat = self.decoder(z)

        return x_hat


model = AMPAutoencoder()

print(model)

AMPAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=1280, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=128, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=1280, bias=True)
  )
)


In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

cuda


In [6]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

print(X_train_tensor.shape)
print(X_val_tensor.shape)

torch.Size([7560, 1280])
torch.Size([840, 1280])


In [7]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor)
val_dataset = TensorDataset(X_val_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

print(len(train_loader))
print(len(val_loader))

30
4


In [8]:
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Ready")

Ready


In [9]:
num_epochs = 30

for epoch in range(num_epochs):

    # -------------------
    # Train
    # -------------------
    model.train()

    train_loss = 0

    for batch in train_loader:

        x = batch[0].to(device)

        optimizer.zero_grad()

        x_hat = model(x)

        loss = criterion(x_hat, x)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------------------
    # Validation
    # -------------------
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            x = batch[0].to(device)

            x_hat = model(x)

            loss = criterion(x_hat, x)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}"
    )

Epoch [1/30] | Train Loss: 0.024597 | Val Loss: 0.007675
Epoch [2/30] | Train Loss: 0.006987 | Val Loss: 0.006800
Epoch [3/30] | Train Loss: 0.006094 | Val Loss: 0.004963
Epoch [4/30] | Train Loss: 0.004392 | Val Loss: 0.004293
Epoch [5/30] | Train Loss: 0.004077 | Val Loss: 0.003924
Epoch [6/30] | Train Loss: 0.003495 | Val Loss: 0.003230
Epoch [7/30] | Train Loss: 0.002994 | Val Loss: 0.002849
Epoch [8/30] | Train Loss: 0.002750 | Val Loss: 0.002680
Epoch [9/30] | Train Loss: 0.002610 | Val Loss: 0.002535
Epoch [10/30] | Train Loss: 0.002535 | Val Loss: 0.002415
Epoch [11/30] | Train Loss: 0.002409 | Val Loss: 0.002273
Epoch [12/30] | Train Loss: 0.002306 | Val Loss: 0.002174
Epoch [13/30] | Train Loss: 0.002215 | Val Loss: 0.002077
Epoch [14/30] | Train Loss: 0.002130 | Val Loss: 0.002006
Epoch [15/30] | Train Loss: 0.002011 | Val Loss: 0.001916
Epoch [16/30] | Train Loss: 0.001899 | Val Loss: 0.001799
Epoch [17/30] | Train Loss: 0.001800 | Val Loss: 0.001695
Epoch [18/30] | Train L

In [10]:
import torch

torch.save(
    model.state_dict(),
    "amp_autoencoder.pt"
)

print("Autoencoder Saved")

Autoencoder Saved


In [11]:
model.eval()

X_tensor = torch.tensor(
    X,
    dtype=torch.float32
).to(device)

with torch.no_grad():

    latent_vectors = (
        model.encoder(X_tensor)
        .cpu()
        .numpy()
    )

print(latent_vectors.shape)

(8400, 128)


In [12]:
import numpy as np

np.save(
    "amp_latent_vectors.npy",
    latent_vectors
)

print("Latent vectors saved")

Latent vectors saved


In [13]:
latent_vectors.shape

(8400, 128)